# Kubeflow SDK: From Raw Data to Training with SparkClient and TrainerClient

This notebook demonstrates the Kubeflow SDK as one flow from data engineering to training. `SparkClient` enriches raw data and writes an Iceberg table, then `TrainerClient` fine-tunes a model that streams that table from the Trainer data cache.

### What you will learn:
1. Setting up `SparkClient` for a Spark Connect session with S3A and Iceberg support.
2. Creating noisy raw reviews in S3-compatible storage (SeaweedFS).
3. Enriching the raw data with Spark and writing an Iceberg table.
4. Capturing the Iceberg metadata location from Spark.
5. Defining a training function that reads from the data cache.
6. Creating a TrainJob with `DataCacheInitializer` using that metadata location.
7. Following the TrainJob logs and verifying completion.
8. Deleting the TrainJob after execution.

### Prerequisites:
- A Kind cluster with the Kubeflow Trainer controller, the Spark Operator and SeaweedFS (bucket `mlpipeline`).
- A Spark image with the S3A and Iceberg jars, loaded into Kind: `kind load docker-image spark-s3-iceberg:4.0.4 --name kubeflow-local` (see `image/Dockerfile`).
- A kernel with the Kubeflow SDK installed (`kubeflow.spark` and `kubeflow.trainer`).

## 1. Imports and Client Setup

Import the Spark and Trainer classes and initialize `SparkClient` targeting your cluster namespace.

In [1]:
import os
import random

from pyspark.sql import functions as fn

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import Driver, Executor, SparkClient
from kubeflow.trainer import CustomTrainer, DataCacheInitializer, Initializer, TrainerClient

# Namespace configuration (uses SPARK_TEST_NAMESPACE if set, else 'default')
namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)

# Initialize SparkClient
spark_client = SparkClient(backend_config=backend_config)
print(f"SparkClient initialized for namespace: {namespace}")

SparkClient initialized for namespace: default


## 2. Configure the Spark Session

Define the Spark image and the S3A and Iceberg catalog settings that point to SeaweedFS, plus a helper that opens and closes a session.

In [2]:
IMAGE = "spark-s3-iceberg:4.0.4"

SPARK_CONF = {
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.sql.catalog.lake": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.lake.type": "hadoop",
    "spark.sql.catalog.lake.warehouse": "s3a://mlpipeline/warehouse",
    "spark.hadoop.fs.s3a.endpoint": "http://seaweedfs-s3.seaweedfs.svc.cluster.local:8333",
    "spark.hadoop.fs.s3a.access.key": "dummy",
    "spark.hadoop.fs.s3a.secret.key": "dummy",
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    "spark.hadoop.fs.s3a.endpoint.region": "us-east-1",
}


def open_session():
    """Create a Spark Connect session with the S3A and Iceberg settings."""
    return spark_client.connect(
        driver=Driver(image=IMAGE, resources={"cpu": "1", "memory": "1Gi"}),
        executor=Executor(num_instances=1, resources_per_executor={"cpu": "1", "memory": "1Gi"}),
        spark_conf=SPARK_CONF,
        timeout=300,
    )


def close_session(spark):
    """Stop the session and delete its SparkConnect resource to free cluster memory."""
    spark.stop()
    for session in spark_client.list_sessions():
        spark_client.delete_session(session.name)


print("Spark session helpers defined successfully.")

Spark session helpers defined successfully.


## 3. Create the Raw Data

In a real setup the raw reviews already exist in S3. This optional step creates synthetic ones with realistic noise: HTML tags, missing text, spam, very short text, duplicates and star ratings from 1 to 5. Skip it if the data is already there.

In [3]:
random.seed(7)
phrases = {
    1: ["Terrible quality, broke on day one.", "Worst purchase ever.", "Complete waste of money."],
    2: ["Not great, feels cheap.", "Disappointed, works only sometimes."],
    3: ["It's okay, nothing special.", "Average product for the price."],
    4: ["Good value, works well.", "Happy with this, would buy again."],
    5: ["Excellent! Exceeds expectations.", "Love it, works perfectly."],
}
rows = []
for i in range(400):
    rating = random.choice([1, 2, 3, 4, 5])
    suffix = random.choice(["Fast shipping.", "Packaging was fine.", "Arrived on time.", ""])
    text = random.choice(phrases[rating]) + " " + suffix
    r = random.random()
    if r < 0.08:
        text = f"<p>{text}</p><br/>"  # HTML noise
    elif r < 0.12:
        text = None  # missing text
    elif r < 0.16:
        text = "BUY NOW cheap deals http://spam.example.com"  # spam
    elif r < 0.19:
        text = "ok"  # too short
    rows.append((f"r{i}", rating, text))
rows += [(f"d{i}", x[1], x[2]) for i, x in enumerate(rows[:10])]  # duplicates

print("Writing raw reviews to s3a://mlpipeline/raw_reviews/ ...")
spark = open_session()
try:
    df = spark.createDataFrame(rows, "review_id string, rating int, review_text string")
    df.coalesce(1).write.mode("overwrite").json("s3a://mlpipeline/raw_reviews/")
    raw_count = df.count()
finally:
    close_session(spark)

assert raw_count > 0, "Expected raw reviews to be written"
print(f"Raw rows written: {raw_count}")

Writing raw reviews to s3a://mlpipeline/raw_reviews/ ...
Raw rows written: 410


## 4. Enrich the Data and Write an Iceberg Table

The TrainJob expects two columns: `content` (string) and `label` (int64, 0 to 4, because the model has `num_labels=5`). Spark strips HTML, drops missing, short and spam rows, removes duplicates, maps the rating 1 to 5 onto the label 0 to 4, and shuffles the rows so each cache shard gets a mix of labels.

In [4]:
print("Enriching raw reviews with Spark...")
spark = open_session()
try:
    raw = spark.read.json("s3a://mlpipeline/raw_reviews/")
    clean = (
        raw.withColumn(
            "content",
            fn.trim(fn.regexp_replace(fn.regexp_replace("review_text", "<[^>]+>", " "), r"\s+", " ")),
        )
        .filter(fn.col("content").isNotNull() & (fn.length("content") >= 10))
        .filter(~fn.lower("content").rlike("http|buy now"))
        .dropDuplicates(["content"])
        .withColumn("label", (fn.col("rating") - 1).cast("long"))
        .filter(fn.col("label").between(0, 4))
        .select("content", "label")
        .orderBy(fn.rand(7))
    )
    raw_rows = raw.count()
    enriched_rows = clean.count()
    clean.groupBy("label").count().orderBy("label").show()

    clean.writeTo("lake.default.reviews_enriched_nb").createOrReplace()

    METADATA_LOC = (
        spark.sql(
            "SELECT file FROM lake.default.reviews_enriched_nb.metadata_log_entries "
            "ORDER BY timestamp DESC LIMIT 1"
        )
        .collect()[0]["file"]
    )
finally:
    close_session(spark)

assert 0 < enriched_rows < raw_rows, f"Expected fewer enriched rows than raw rows, got {enriched_rows}"
assert METADATA_LOC.endswith(".metadata.json"), f"Unexpected metadata location: {METADATA_LOC}"
print(f"Raw rows: {raw_rows} -> enriched rows: {enriched_rows}")
print(f"METADATA_LOC: {METADATA_LOC}")

Enriching raw reviews with Spark...
+-----+-----+
|label|count|
+-----+-----+
|    0|   12|
|    1|    8|
|    2|    8|
|    3|    8|
|    4|    8|
+-----+-----+

Raw rows: 410 -> enriched rows: 44
METADATA_LOC: s3a://mlpipeline/warehouse/default/reviews_enriched_nb/metadata/v1.metadata.json


# SparkClient: Python Function (FuncJob) Lifecycle

This notebook demonstrates how to submit and manage an in-memory Python function as a distributed Spark job using `FuncJob` and `SparkClient` from the Kubeflow SDK.

### What you will learn:
1. Defining a native Python function for execution on Spark.
2. Submitting the function via `FuncJob` with custom arguments and executor resources.
3. Polling and waiting for the job to complete.
4. Inspecting job metadata and driver pod information.
5. Listing active and completed jobs.
6. Streaming driver pod logs to observe function output.
7. Deleting the Spark job resource after execution.

## 1. Imports and Client Setup

Import necessary classes from `kubeflow.spark` and initialize `SparkClient` targeting your cluster namespace.

In [5]:
import os

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import FuncJob, SparkClient, SparkJobStatus

# Namespace configuration (uses SPARK_TEST_NAMESPACE if set, else 'default')
namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)

# Initialize SparkClient
client = SparkClient(backend_config=backend_config)
print(f"SparkClient initialized for namespace: {namespace}")

SparkClient initialized for namespace: default


## 2. Define the Spark Function

Define the Python function to be executed remotely. Inside the function, initialize a `SparkSession`, perform computations, and terminate the session.

In [6]:
def estimate_pi(samples: int) -> None:
    """Simple function executed as a Spark FuncJob."""
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("estimate-pi").getOrCreate()
    count = spark.range(samples).count()
    print(f"Estimated Pi over {count} samples.")
    spark.stop()


print("Spark function 'estimate_pi' defined successfully.")

Spark function 'estimate_pi' defined successfully.


## 3. Submit the FuncJob

Package the function into a `FuncJob` and submit it with executor configuration.

In [7]:
print("Submitting Spark FuncJob...")
job_name = client.submit_job(
    job=FuncJob(
        func=estimate_pi,
        func_args={
            "samples": 10,
        },
    ),
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512Mi",
    },
)

print(f"Job submitted successfully: {job_name}")

Submitting Spark FuncJob...
Job submitted successfully: spark-job-6b996361


## 4. Wait for Completion

Block until the job reaches completion or the timeout expires.

In [8]:
print(f"Waiting for {job_name} to complete...")
job = client.wait_for_job_status(
    job_name,
    timeout=300,
)
assert job is not None, f"Expected job {job_name} to exist"
assert job.status == SparkJobStatus.COMPLETED, f"Expected COMPLETED, got {job.status}"

print("Job completed successfully.")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Namespace: {job.namespace}")

Waiting for spark-job-6b996361 to complete...
Job completed successfully.
Status: SparkJobStatus.COMPLETED
Driver Pod: spark-job-6b996361-driver
Namespace: default


## 5. Retrieve Job Metadata

Inspect job details returned by the Kubernetes Spark Operator.

In [9]:
job_info = client.get_job(job_name)
assert job_info is not None, f"Failed to retrieve metadata for {job_name}"
assert job_info.status == SparkJobStatus.COMPLETED, (
    f"Expected COMPLETED status, got {job_info.status}"
)

print(f"Name: {job_info.name}")
print(f"Namespace: {job_info.namespace}")
print(f"Status: {job_info.status}")
print(f"Driver Pod: {job_info.driver_pod_name}")
print(f"Executors: {job_info.num_executors}")

Name: spark-job-6b996361
Namespace: default
Status: SparkJobStatus.COMPLETED
Driver Pod: spark-job-6b996361-driver
Executors: 1


## 6. List Spark Jobs

List all jobs in the namespace and verify completed status filtering.

In [10]:
jobs = client.list_jobs()
print(f"Found {len(jobs)} total Spark job(s).")
for j in jobs:
    print(f"- {j.name} | Status: {j.status} | Namespace: {j.namespace}")

completed_jobs = client.list_jobs(status={SparkJobStatus.COMPLETED})
print(f"\nFound {len(completed_jobs)} completed Spark job(s).")

assert any(j.name == job_name for j in jobs), f"Expected {job_name} in total jobs list"
assert any(j.name == job_name for j in completed_jobs), (
    f"Expected {job_name} in completed jobs list"
)

Found 1 total Spark job(s).
- spark-job-6b996361 | Status: SparkJobStatus.COMPLETED | Namespace: default

Found 1 completed Spark job(s).


## 7. Stream Driver Logs

Inspect the first 20 lines of the driver pod logs to confirm the output of `estimate_pi`.

In [11]:
print(f"Retrieving logs for {job_name} (first 20 lines):")
print("-" * 70)

line_count = 0
for line in client.get_job_logs(job_name):
    print(line.rstrip())
    line_count += 1
    if line_count >= 20:
        print("...")
        break
print("-" * 70)
assert line_count > 0, f"Expected driver logs to be present for {job_name}"

Retrieving logs for spark-job-6b996361 (first 20 lines):
----------------------------------------------------------------------
26/11/23 07:28:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/23 07:28:41 INFO SparkContext: Running Spark version 4.0.4
26/11/23 07:28:41 INFO SparkContext: OS info Linux, 6.6.114.1-microsoft-standard-WSL2, amd64
26/11/23 07:28:41 INFO SparkContext: Java version 17.0.19
26/11/23 07:28:41 INFO ResourceUtils: ==============================================================
26/11/23 07:28:41 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/23 07:28:41 INFO ResourceUtils: ==============================================================
26/11/23 07:28:41 INFO SparkContext: Submitted application: estimate-pi
26/11/23 07:28:41 INFO ResourceProfile: Default ResourceProfile created

## 8. Clean Up Resources

Delete the completed Spark application resource.

In [12]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Job deletion requested successfully.")

Deleting job: spark-job-6b996361
Job deletion requested successfully.


## 5. Define the Training Function

The training function reads its shard from the data cache over Arrow Flight and fine-tunes DistilBERT on the `content` and `label` columns.

In [13]:
def train_func():
    import os
    import time

    import pyarrow as pa
    import pyarrow.flight
    import torch
    import torch.distributed as dist
    from pyarrow._flight import FlightClient
    from torch.utils.data import IterableDataset
    from transformers import AdamW, DistilBertForSequenceClassification, DistilBertTokenizer

    class DataCacheDataset(IterableDataset):
        def __init__(self, seed=0, batch_size=128):
            train_job_name = os.getenv("TRAIN_JOB_NAME")
            self.endpoint = f"grpc://{train_job_name}-cache-service:50051"
            self.seed = seed
            self.batch_size = batch_size

        def set_epoch(self, epoch):
            self.seed = self.seed + epoch

        def from_arrow_rb_to_tensor(self, chunk):
            return

        def stream_recordbatch_chunks(self, recordbatch_stream, chunk_size):
            leftover = None
            for rb in recordbatch_stream:
                batch = rb.data
                if leftover is not None:
                    arrays = [
                        pa.concat_arrays([leftover.column(i), batch.column(i)])
                        for i in range(len(batch.columns))
                    ]
                    batch = pa.RecordBatch.from_arrays(arrays, batch.schema.names)
                for i in range(0, batch.num_rows, chunk_size):
                    chunk = batch.slice(i, chunk_size)
                    if chunk.num_rows < chunk_size:
                        leftover = chunk
                        break
                    yield from self.from_arrow_rb_to_tensor(chunk)
                else:
                    leftover = None
            if leftover is not None and leftover.num_rows > 0:
                yield from self.from_arrow_rb_to_tensor(leftover)

        def __iter__(self):
            worker_info = torch.utils.data.get_worker_info()
            if worker_info is None:
                worker_id, num_workers = 0, 1
            else:
                worker_id, num_workers = worker_info.id, worker_info.num_workers

            world_size = dist.get_world_size()
            rank = dist.get_rank()
            total_workers = num_workers * world_size
            global_worker_id = worker_id * world_size + rank
            index = (global_worker_id + self.seed) % total_workers

            descriptor = pa.flight.FlightDescriptor.for_path(str(index), str(total_workers))
            flight_client = pa.flight.connect(self.endpoint)
            flight_info = flight_client.get_flight_info(descriptor)
            for endpoint in flight_info.endpoints:
                for location in endpoint.locations:
                    _client = FlightClient(location.uri)
                    reader = _client.do_get(endpoint.ticket)
                    yield from self.stream_recordbatch_chunks(reader, self.batch_size)

    class ReviewDataset(DataCacheDataset):
        def __init__(self, tokenizer, seed=0):
            super().__init__(seed)
            self.tokenizer = tokenizer
            self.row_count = 0

        def from_arrow_rb_to_tensor(self, chunk):
            self.row_count += chunk.num_rows
            texts = chunk.column("content").to_pylist()
            labels = chunk.column("label").to_pylist()
            encoding = self.tokenizer(
                texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt"
            )
            # No squeeze(0): it flattens a one-row batch and breaks the model call.
            yield {
                "input_ids": encoding["input_ids"],
                "attention_mask": encoding["attention_mask"],
                "labels": torch.tensor(labels),
            }

    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    device = torch.device(f"cuda:{local_rank}" if device == "cuda" else "cpu")

    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=5)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

    start = time.time()
    dataset = ReviewDataset(tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=None)

    for epoch in range(1):
        dataset.set_epoch(epoch)
        for batch in dataloader:
            outputs = model(
                batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            if dist.get_rank() == 0:
                print(f"Epoch {epoch}, Loss: {loss.item()}")

    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")
    print(f"Time taken: {time.time() - start:.6f} seconds")
    print(f"Total rows processed: {dataset.row_count}")
    dist.destroy_process_group()


print("Training function 'train_func' defined successfully.")

Training function 'train_func' defined successfully.


## 6. Create the TrainJob

Initialize `TrainerClient`, find the runtime that includes the data cache, and submit the TrainJob. `DataCacheInitializer` loads the Iceberg table written by Spark, using the metadata location captured above.

In [14]:
trainer_client = TrainerClient()
cache_runtime = next(
    (r for r in trainer_client.list_runtimes() if r.name == "torch-distributed-with-cache"), None
)
assert cache_runtime is not None, "Runtime 'torch-distributed-with-cache' not found"

print("Submitting TrainJob...")
job_name = trainer_client.train(
    trainer=CustomTrainer(
        func=train_func,
        num_nodes=1,
        resources_per_node={"cpu": 1, "memory": "4Gi"},
        packages_to_install=["pyarrow==19.0.0", "transformers==4.45.2"],
    ),
    initializer=Initializer(
        dataset=DataCacheInitializer(
            num_data_nodes=2,
            metadata_loc=METADATA_LOC,
            storage_uri="cache://default/reviews_enriched",
            iam_role="dummy",
        )
    ),
    runtime=cache_runtime,
)

print(f"TrainJob created: {job_name}")

Submitting TrainJob...
TrainJob created: d2dd51209bd8


## 7. Stream the TrainJob Logs

Wait for the job to start, then follow its logs. On one CPU the first run takes several minutes: it downloads the model, then trains on all rows. The run should end with `Training is finished` and a row count equal to the enriched row count from Spark.

In [15]:
trainer_client.wait_for_job_status(job_name, status={"Running"})
print(f"{job_name} is running. Following logs:")
print("-" * 70)
for line in trainer_client.get_job_logs(job_name, follow=True):
    print(line)
print("-" * 70)

d2dd51209bd8 is running. Following logs:
----------------------------------------------------------------------
Using Device: cpu, Backend: gloo
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/root/.local/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 0, Loss: 1.6057807207107544
Training is finished
Time taken: 126.788975 seconds
Total rows processed: 44
----------------------------------------------------------------------


## 8. Wait for Completion

Block until the TrainJob completes or the timeout expires.

In [16]:
print(f"Waiting for {job_name} to complete...")
trainer_client.wait_for_job_status(job_name, timeout=300)
print("TrainJob completed successfully.")

Waiting for d2dd51209bd8 to complete...
TrainJob completed successfully.


## 9. Clean Up Resources

Delete the TrainJob. The Spark sessions were already deleted in the steps above.

In [17]:
print(f"Deleting TrainJob: {job_name}")
trainer_client.delete_job(job_name)
print("TrainJob deletion requested successfully.")

Deleting TrainJob: d2dd51209bd8
TrainJob deletion requested successfully.
